# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">9/29(화) 오후 · 탐지 룰 ③ · API·HTTP — 실습</mark>

오전에 수상한 IP 하나(`185.220.101.34`)와 두드려 맞은 계정 하나(`admin`)를 찾았습니다.

오후는 그 두 발견에서 출발합니다. **성공을 의심하고 → 밖에 물어보고 → 답이 안 올 때를 읽습니다.**


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 내 사본 만들기

위 메뉴에서 파일 › 드라이브에 사본 저장을 누릅니다. 제목이 「사본: …」으로 바뀌면 된 것입니다.

### 0.2 오늘 오후의 순서

| 교시 | 무엇 |
|---|---|
| 5교시 | 룰 ③ 심야 성공 접속 · 세 룰을 한 번에 |
| 6교시 | 왜 밖에 묻나 · URL 구조와 메서드 |
| 7교시 | 상태코드 · 401 ↔ 403 · 인증 두 방식 |
| 8교시 | 확인 · 남은 도전 · 내일 예고 |

### 0.3 막혔을 때

1. 문제 아래 **💡 힌트**를 순서대로 따라 합니다.
2. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다.
3. 도전 문제는 안 풀고 넘어가도 됩니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">5교시 (14:00–14:50) · 탐지 룰 ③</mark>


아래 두 셀을 먼저 실행합니다. 오전 파일을 여기서 다시 만듭니다. **오전을 못 끝냈어도 여기서부터 시작할 수 있습니다.**


In [ ]:
%%writefile raw_logs.txt
2026-09-29 09:02:11 INFO accepted login for kim.cs from 10.1.2.11
2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5
2026-09-29 09:15:31 INFO accepted login for lee.yh from 10.1.2.34
2026-09-29 09:16:02 INFO session closed for 10.1.2.34
2026-09-29 10:03:19 WARN failed login for park.js from 10.1.3.7
2026-09-29 10:03:31 INFO accepted login for park.js from 10.1.3.7
2026-09-29 11:20:55 INFO accepted login for choi.mk from 10.1.4.2
2026-09-29 12:40:12 INFO session closed for 10.1.4.2
2026-09-29 03:11:05 WARN failed login for admin from 211.45.12.9
2026-09-29 03:12:47 WARN failed login for admin from 211.45.12.9
2026-09-29 03:13:58 WARN failed login for admin from 211.45.12.9
2026-09-29 03:15:22 WARN failed login for admin from 211.45.12.9
2026-09-29 03:17:09 INFO accepted login for admin from 211.45.12.9
2026-09-29 14:05:38 INFO accepted login for jung.hw from 10.1.2.88
2026-09-29 22:14:03 WARN failed login for kim.cs from 185.220.101.34
2026-09-29 22:14:21 WARN failed login for lee.yh from 185.220.101.34
2026-09-29 22:14:40 WARN failed login for choi.mk from 185.220.101.34
2026-09-29 22:14:58 WARN failed login for jung.hw from 185.220.101.34
2026-09-29 22:15:12 INFO session closed for 185.220.101.34
2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34


Writing raw_logs.txt


In [ ]:
import re
import json

PATTERN = r"(?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

rows = []
unmatched = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            rows.append(m.groupdict())
        else:
            unmatched.append(line.strip())

with open("normalized_logs.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"정규화 {len(rows)}건 / 안 맞음 {len(unmatched)}건")


정규화 16건 / 안 맞음 4건


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **탐지 룰** | 보안 관제에서 룰을 만든다는 것이 무슨 일인가 |
| **심야 접속** | 새벽의 로그인 성공을 왜 의심하나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 탐지 룰 →
- 심야 접속 →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4 · 실패가 아니라 성공을 의심한다</mark>


### 왜 필요한가

1. 오전의 룰 두 개는 모두 **실패**를 셌습니다. 실패가 잦으면 수상하다는 기준이었습니다.
2. 그런데 이 로그에는 실패 네 번 뒤에 **성공**이 한 번 있습니다. 새벽 3시 17분, 계정은 `admin` 입니다.
3. 실패는 막힌 것이고 **성공은 뚫린 것**입니다. 우리 룰은 뚫린 쪽을 못 보고 있었습니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 룰 ③ 심야 접속 | 사람이 자는 시간대의 **로그인 성공**을 의심한다 |
| 업무 시간 | 정상이라고 보는 시간대. 몇 시부터 몇 시까지인지는 사람이 정한다 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.1 룰 ③ — 시간대로 거른다</mark>

시각은 `09:02:11` 처럼 글자로 들어 있습니다. **콜론으로 나눠** 앞 칸을 숫자로 바꿉니다.

```python
time = "03:17:09"
hour = int(time.split(":")[0])
print(hour)          # 3
```

오전 3교시에 쓴 방법과 같습니다. 나누면 `['03', '17', '09']` 가 되고, 0번 칸이 **시**입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
time = "22:14:03"
print(int(time.split(":")[0]))
```

막히면 바로 위 `5.1 룰 ③ — 시간대로 거른다` 설명을 다시 봅니다.


In [ ]:
time = "22:14:03"
print(int(time.split(":")[0]))


22


✅ `22`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 읽어 온 기록에서 성공만 세어 봅니다.

```python
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

ok = 0
for row in rows:
    if row["level"] == "INFO":
        ok = ok + 1

print(ok)
```

막히면 바로 위 `5.1 룰 ③ — 시간대로 거른다` 설명을 다시 봅니다.


In [ ]:
import json

for row in rows:
    if row["level"] == "INFO":
        print(row)
print(ok)


{'time': '09:02:11', 'level': 'INFO', 'user': 'kim.cs', 'ip': '10.1.2.11'}
{'time': '09:15:31', 'level': 'INFO', 'user': 'lee.yh', 'ip': '10.1.2.34'}
{'time': '10:03:31', 'level': 'INFO', 'user': 'park.js', 'ip': '10.1.3.7'}
{'time': '11:20:55', 'level': 'INFO', 'user': 'choi.mk', 'ip': '10.1.4.2'}
{'time': '03:17:09', 'level': 'INFO', 'user': 'admin', 'ip': '211.45.12.9'}
{'time': '14:05:38', 'level': 'INFO', 'user': 'jung.hw', 'ip': '10.1.2.88'}
6


✅ `6`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-3 · 룰 ③ 완성 — 새벽에 성공한 것</font></h3></td></tr></table>

`normalized_logs.json` 을 읽어, **06시 이전**(`시 < 6`)의 기록 중 **성공(`INFO`)** 인 것만 골라 경보를 출력하시오.
시각은 `row["time"]` 이 `"03:17:09"` 꼴이라 앞 두 글자가 시입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[룰 3] 심야 접속: 03:17:09 admin (211.45.12.9)` |

**💡 힌트**

1. `rows` 를 돌면서 `row["time"]` 의 앞 두 글자를 숫자로 바꿉니다 — `int(row["time"].split(":")[0])`
2. 그 시가 6 보다 작고, 등급이 `INFO` 인 것만 고릅니다. 두 조건은 `and` 로 잇습니다.
3. f-string 으로 문장을 만들어 출력합니다.


In [ ]:
# 여기에 코드를 입력하세요
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

for row in rows:
    hour = int(row["time"].split(":")[0])

    if hour < 6 and row["level"] == "INFO":
        print(f'[룰 3] 심야 접속: {row["time"]} {row["user"]} ({row["ip"]})')

[룰 3] 심야 접속: 03:17:09 admin (211.45.12.9)


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-4 · 기준 시각을 맨 위로</font></h3></td></tr></table>

문제 5-3에서 새벽 기준으로 쓴 숫자 `6` 을 **`NIGHT_END = 6`** 으로 맨 위에 올리고, `if` 줄에서는 그 이름을 쓰시오.
기준을 바꿀 때 **한 곳만 고치면 되게** 하려는 것입니다. 오전 룰 ①의 `THRESHOLD` 와 같은 방식입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 문제 5-3와 같다 |

**💡 힌트**

1. 맨 위에 `NIGHT_END = 6` 을 적습니다.
2. `if` 줄의 숫자 자리를 그 이름으로 바꿉니다.
3. 오전 룰 ①의 `THRESHOLD` 와 같은 방식입니다.


In [ ]:
# 여기에 코드를 입력하세요
import json

NIGHT_END = 6

for row in rows:
    hour = int(row["time"].split(":")[0])

    if hour < NIGHT_END and row["level"]=="INFO":
      print(f'[룰 3] 심야접속 : {row["time"]}{row["user"]}{row["ip"]})')

[룰 3] 심야접속 : 03:17:09admin211.45.12.9)


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-5 · 같은 시간대에 몰린 실패 찾기</font></h3></td></tr></table>

`normalized_logs.json` 을 읽어, 같은 계정의 **실패(`WARN`)가 같은 시(時)에 3회 이상** 있었는지 찾아 출력하시오.
시각은 `row["time"]` 이 `"03:11:05"` 꼴이라 앞 두 글자가 시입니다.
짧은 시간에 몰린 실패는 사람이 아니라 프로그램일 가능성이 큽니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[집중 실패] admin 03시 — 4회` |

**💡 힌트**

1. 세는 기준이 둘(계정·시)입니다. 두 글자를 이어 붙여 **키 하나**로 만듭니다 — `row["user"] + " " + 시`
2. 시는 `row["time"].split(":")[0]` 으로 꺼냅니다.
3. 오전에 계정을 셀 때 쓴 딕셔너리 방법 그대로입니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-6 · 기준 시각을 정하고 근거를 쓰기</font></h3></td></tr></table>

문제 5-4의 코드에서 `NIGHT_END` 를 **6 · 9 · 12** 로 각각 돌려 경보가 몇 건이 되는지 보고,
**우리 로그에는 몇이 맞다고 생각하는지 한 줄로 근거를 적으시오.**

- 정답이 하나로 정해져 있지 않습니다. 숫자보다 **왜 그렇게 정했는지**가 중요합니다.
- 기준을 넓힐수록 경보가 늘고, 그중 정상 접속(헛경보)도 같이 늡니다.

| | |
|---|---|
| 🎯 확인 | 세 기준의 경보 건수를 비교하고, 고른 값과 그 이유를 한 줄로 적었다 |

**💡 힌트**

- 문제 5-4의 코드를 그대로 쓰고 맨 위 숫자만 바꿔 가며 돌려 봅니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.2 세 룰을 한 번에</mark>

룰 세 개는 **같은 `rows` 를 세 가지 각도로 보는 것**입니다. 한 번 읽어 두고 셋을 차례로 돌립니다.

| 룰 | 무엇을 본다 | 기준 |
|---|---|---|
| ① 브루트포스 | 계정별 실패 횟수 | 3회 이상 |
| ② 의심 IP | IP 마다 시도한 계정 수 | 2개 이상 |
| ③ 심야 접속 | 새벽의 로그인 성공 | 06시 이전 |

오전에 만든 룰 ①②의 코드를 그대로 가져와 ③ 아래에 붙이면 됩니다.


룰 ①②가 잘 안 되면 **아래 준비 셀을 실행**하고 룰 ③만 이어 붙여도 됩니다. 경보 두 줄을 대신 만들어 줍니다.

In [ ]:
#@title 준비 · 룰 ①② 경보 두 줄 (룰 ①② 가 잘 안 될 때만 실행) { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

THRESHOLD = 3
fail = {}
table = {}
for row in rows:
    if row["level"] == "WARN":
        user = row["user"]
        if user in fail:
            fail[user] = fail[user] + 1
        else:
            fail[user] = 1
    ip = row["ip"]
    if ip in table:
        if row["user"] not in table[ip]:
            table[ip].append(row["user"])
    else:
        table[ip] = [row["user"]]

for user in fail:
    if fail[user] >= THRESHOLD:
        print(f"[룰 1] 확인 필요: {user} — 실패 {fail[user]}회")
for ip in table:
    if len(table[ip]) >= 2:
        print(f"[룰 2] 의심 IP: {ip} — 계정 {len(table[ip])}개 시도")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-7 · 세 줄을 한 번에</font></h3></td></tr></table>

세 룰을 한 코드에 이어 붙여 경보 **세 줄**을 한 번에 출력하시오. 오늘 오후의 첫 도착점입니다.

| 룰 | 기준 | 내는 문장 |
|---|---|---|
| ① 브루트포스 | 계정별 `WARN` 이 `THRESHOLD`(3) 이상 | `[룰 1] 확인 필요: admin — 실패 4회` |
| ② 의심 IP | 한 IP 에서 계정 2개 이상 | `[룰 2] 의심 IP: 185.220.101.34 — 계정 4개 시도` |
| ③ 심야 성공 | `NIGHT_END`(6) 이전의 `INFO` | `[룰 3] 심야 접속: 03:17:09 admin (211.45.12.9)` |

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[룰 1]` · `[룰 2]` · `[룰 3]` 세 줄 |

**💡 힌트**

1. 오전 룰 ①②의 코드를 먼저 두고, 그 뒤에 방금 만든 룰 ③을 이어 붙입니다.
2. 파일을 읽는 줄은 맨 위에 한 번만 둡니다.
3. 기준값 두 개(`THRESHOLD`·`NIGHT_END`)를 맨 위에 나란히 둡니다.
4. 룰 ①②가 잘 안 되면 **바로 아래 준비 셀**을 실행하고 룰 ③만 붙여도 됩니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-8 · 요약 한 줄 붙이기</font></h3></td></tr></table>

경보 세 줄 **위에** 요약 한 줄을 붙이시오.

- `16` 은 읽어 온 기록 수(`len(rows)`), `3` 은 모아 둔 경보 수(`len(alerts)`)입니다. 숫자를 손으로 적지 말고 세어서 넣습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[요약] 정규화 16건 / 경보 3건` 다음에 세 줄 |

**💡 힌트**

1. 경보를 바로 출력하지 말고 리스트에 모읍니다.
2. 모은 뒤 개수를 세어 요약을 먼저 출력합니다.
3. 그다음 리스트를 돌며 한 줄씩 출력합니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-9 · 경보에 심각도 붙이기</font></h3></td></tr></table>

세 룰이 지목한 대상 목록을 받아, **두 개 이상의 룰에 걸린 대상은 `HIGH`**, 하나뿐이면 `MEDIUM` 으로 출력하시오.
여러 룰에 동시에 걸릴수록 진짜일 가능성이 큽니다.

| | |
|---|---|
| 주어지는 값 | `targets = ["admin", "185.220.101.34", "admin"]` |
| 🎯 나와야 하는 결과 | `[HIGH] admin — 2개 룰` · `[MEDIUM] 185.220.101.34 — 1개 룰` |

**💡 힌트**

1. 몇 번 나왔는지 세면 됩니다. 세는 방법은 문제 5-5와 같습니다.
2. 2 이상인지 아닌지로 갈라 출력합니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-10 · 두 룰에 함께 걸린 계정 찾기</font></h3></td></tr></table>

세 룰은 각각 **무엇을 내놓는지가 다릅니다.**

| 룰 | 무엇을 보나 | 걸리면 내놓는 값 |
|---|---|---|
| 룰 ① 브루트포스 | 계정별 실패 횟수가 `THRESHOLD` 이상 | **계정 이름** — 예: `admin` |
| 룰 ② 의심 IP | 한 IP 에서 계정 2개 이상 | **IP** — 예: `185.220.101.34` |
| 룰 ③ 심야 성공 | `NIGHT_END` 이전의 `INFO` | **계정 이름** — 예: `admin` |

룰 ①과 룰 ③은 **둘 다 계정 이름**을 내놓습니다. 그래서 **같은 계정이 양쪽에 다 나올 수 있습니다.**
낮에 비밀번호를 여러 번 틀린 계정이 새벽에 성공까지 했다면, 한쪽에만 걸린 것보다 훨씬 수상합니다.

**할 일** — 룰 ①이 내놓은 계정 목록과 룰 ③이 내놓은 계정 목록을 각각 만들고,
**두 목록에 다 들어 있는 계정**을 찾아 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 룰 ①의 목록, 룰 ③의 목록, 그리고 `두 룰에 함께 걸림: admin` 한 줄 |

**💡 힌트**

- 두 목록을 따로 만든 뒤, 한쪽을 돌면서 **다른 쪽에 있는지** 확인하면 됩니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `int(time.split(":")[0])` | 글자 시각에서 **시**만 숫자로 꺼낸다 |
| `hour < NIGHT_END` | 새벽인지 본다 |
| `level == "INFO"` | 성공한 접속인지 본다 |
| 기준값을 맨 위에 | 바꿀 때 한 곳만 고친다 |

**경보는 파일로 남기지 않습니다.** 내일 필요하면 `normalized_logs.json` 을 다시 읽어 룰을 한 번 더 돌립니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">6교시 (15:00–15:50) · API 와 HTTP</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **API** | 남의 프로그램에 물어본다는 것이 어떤 구조인가 |
| **HTTP 요청과 응답** | 한 번 묻고 한 번 받는다는 것이 무슨 뜻인가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- API →
- HTTP 요청과 응답 →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5 · 우리 파일 밖으로 나간다</mark>


### 왜 필요한가

1. 경보 세 줄을 띄웠습니다. 그런데 `185.220.101.34` 가 **정말 위험한 곳인지**는 우리 파일 어디에도 없습니다.
2. 바깥의 IP 조회 서비스에 물어봐야 합니다. 남의 컴퓨터에 들어가 볼 수는 없습니다.
3. 그래서 물어볼 수 있는 것을 미리 정해 **창구**로 열어 둡니다. 식당의 메뉴판과 같습니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| API | 남의 프로그램에 무엇을 물어볼지 정해 둔 창구 |
| HTTP | 그 창구에서 오가는 말의 규칙 |
| 요청 | 이쪽이 보내는 말 |
| 응답 | 저쪽이 돌려주는 말 |
| 메서드 | 무엇을 하려는지 밝히는 표시 (`GET`·`POST`) |


> **오늘은 실제로 요청을 보내지 않습니다.** 보내는 도구(`requests`)는 내일 배웁니다.
> 오늘은 **서버가 줬다고 치는 값**을 가지고 읽고 판단하는 연습만 합니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.1 응답은 글자로 온다</mark>

서버가 돌려주는 것은 **글자 한 덩이**입니다. 어제 배운 `json.loads` 로 값으로 되돌립니다.

```python
import json

text = '{"city": "seoul", "count": 3}'
data = json.loads(text)

print(data["city"])        # seoul
print(data["count"] + 1)   # 4   ← 숫자는 숫자로 돌아온다
```

되돌린 결과는 **딕셔너리**입니다. 그래서 이름으로 바로 꺼낼 수 있습니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
import json

response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'
data = json.loads(response_text)

print(data["country"])
```

막히면 바로 위 `6.1 응답은 글자로 온다` 설명을 다시 봅니다.


In [3]:
import json

response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'
data = json.loads(response_text)

print(data["country"])


Germany


✅ `Germany`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 **숫자 칸**을 꺼내 1을 더합니다. `asn` 은 IP 를 배정받은 회사에 붙은 번호입니다.

```python
import json

response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'
data = json.loads(response_text)

print(data["asn"] + 1)
```

막히면 바로 위 `6.1 응답은 글자로 온다` 설명을 다시 봅니다.


In [2]:
import json

response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'
data = json.loads(response_text)

print(data["asn"] + 1)


60730


✅ `60730`


글자 안에 있던 `60729` 가 **숫자로** 돌아왔습니다. 그래서 더하기가 됩니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-3 · 나라 코드 꺼내기</font></h3></td></tr></table>

응답에서 **나라 코드**를 꺼내 출력하시오.

| | |
|---|---|
| 주어지는 값 | `response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'` |
| 🎯 나와야 하는 결과 | `DE` |

**💡 힌트**

1. `json.loads` 로 먼저 값으로 되돌립니다.
2. 되돌린 것은 딕셔너리입니다.
3. 칸 이름은 `country_code` 입니다.


In [4]:
import json

response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'
data = json.loads(response_text)
print(data["country_code"])

DE


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-4 · 국내가 아니면 경보</font></h3></td></tr></table>

나라 코드가 `KR` 이 **아니면** 경보 한 줄을 출력하시오.

| | |
|---|---|
| 주어지는 값 | `response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'` |
| 🎯 나와야 하는 결과 | `[조회] 185.220.101.34 — 해외 접속 (Germany)` |

**💡 힌트**

1. 되돌린 뒤 `if` 로 견줍니다.
2. **같지 않은지** 볼 때는 `!=` 입니다.
3. f-string 으로 IP 와 나라 이름을 함께 냅니다.


In [6]:
import json

response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'
data = json.loads(response_text)

if data["country_code"] != "KR":
  print(f"[조회] {data['ip']} — 해외 접속 ({data['country']})")

[조회] 185.220.101.34 — 해외 접속 (Germany)


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-5 · 나라에 따라 다르게 판단하기</font></h3></td></tr></table>

나라 코드가 `KR` 이면 `국내 접속`, 아니면 `해외 접속 — 확인 필요` 를 출력하시오.

| | |
|---|---|
| 주어지는 값 | `response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'` |
| 🎯 나와야 하는 결과 | `해외 접속 — 확인 필요` |

**💡 힌트**

1. 앞 문제와 달리 **두 갈래**라서 `else:` 가 필요합니다.
2. `if` 아래에 `else:` 를 같은 들여쓰기로 둡니다.
3. 기준값 `"KR"` 은 맨 위에 한 번만 적어 둡니다.


In [7]:
import json

response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'
HOME_COUNTRY = "KR"
data = json.loads(response_text)

if data["country_code"] == HOME_COUNTRY:
    print("국내 접속")
else:
    print("해외 접속 - 확인 필요")

해외 접속 - 확인 필요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-6 · 없는 값을 안전하게 읽기</font></h3></td></tr></table>

응답 두 개를 차례로 읽어 **IP 와 도시**를 한 줄씩 출력하시오.
그런데 **둘째 응답에는 `city` 가 아예 없습니다.** 없는 키를 그냥 꺼내면 거기서 프로그램이 멈춥니다.

**그래서 이렇게 합니다 — 꺼내기 전에 그 키가 있는지 먼저 확인하고, 두 갈래로 나눠 출력합니다.**

| 그 응답에 `city` 가 | 출력할 것 |
|---|---|
| 있으면 | `IP — 도시이름` |
| 없으면 | `IP — 도시 정보 없음` |

| | |
|---|---|
| 주어지는 값 | `answers = ['{"ip": "185.220.101.34", "country": "Germany", "city": "Berlin"}', '{"ip": "8.8.8.8", "country": "United States"}']` |
| 🎯 나와야 하는 결과 | `185.220.101.34 — Berlin` 과 `8.8.8.8 — 도시 정보 없음` 두 줄 |

**💡 힌트**

1. 글자를 딕셔너리로 되돌리는 것은 `json.loads` 입니다.
2. 키가 있는지는 `if "city" in data:` 로 확인합니다. 없을 때는 `else` 로 받습니다.
3. 바깥 서비스의 응답은 **늘 같은 모양으로 오지 않습니다.** 확인하고 쓰는 습관이 여기서 생깁니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-7 · 응답 세 개를 한 번에</font></h3></td></tr></table>

응답 글자 세 개를 리스트에 담고, **국내(`KR`)가 아닌 것만** 골라 IP 를 출력하시오.

| | |
|---|---|
| 주어지는 값 | `responses = ['{"ip": "185.220.101.34", "country_code": "DE"}', '{"ip": "10.1.2.11", "country_code": "KR"}', '{"ip": "203.0.113.5", "country_code": "US"}']` |
| 🎯 나와야 하는 결과 | `185.220.101.34` · `203.0.113.5` |

**💡 힌트**

- 리스트에 글자 그대로 담아 두고 하나씩 되돌리면 됩니다.


In [ ]:
# 여기에 코드를 입력하세요


### 6.2 주소를 쪼개 읽는다 · HTTP 메서드

요청을 보낼 주소는 네 조각으로 되어 있습니다.

```
https://api.example.com/v1/ip/185.220.101.34?full=true
```

| 조각 | 이 주소에서는 | 무엇 |
|---|---|---|
| 방식 | `https://` | 어떻게 주고받을지 — `https` 는 암호화해서 주고받는다는 뜻 |
| 호스트 | `api.example.com` | 어느 서버에게 |
| 경로 | `/v1/ip/185.220.101.34` | 그 서버의 어느 자리에. 조회할 IP 가 여기 들어간다 |
| 질의 | `?full=true` | 덧붙이는 조건. `?` 뒤가 질의, 여러 개면 `&` 로 잇는다 |

**HTTP 메서드**는 그 주소에 무엇을 하려는지 밝히는 표시입니다. 파이썬 메서드(`.split()`)와 이름만 같고 다른 것으로, 이쪽은 **HTTP 규격이 정한 이름**입니다.

| HTTP 메서드 | 무엇 | 다시 보내면 |
|---|---|---|
| `GET` | 조회한다 | 같은 값을 한 번 더 읽어 온다 — **안전하다** |
| `POST` | 새로 만든다 | 주문·가입이 **한 건 더** 만들어진다 — **사고가 된다** |

오늘 우리가 하려는 일은 **IP 조회**라서 `GET` 입니다. 실패하면 그냥 다시 보내면 됩니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-8 · 질의만 꺼내기</font></h3></td></tr></table>

주소에서 **질의 부분**만 꺼내 출력하시오.

| | |
|---|---|
| 주어지는 값 | `url = "https://api.example.com/v1/ip/185.220.101.34?full=true"` |
| 🎯 나와야 하는 결과 | `full=true` |

**💡 힌트**

1. **주소를** `url.split("?")` 로 쪼개면 두 조각이 됩니다.
2. 앞 조각이 주소, 뒤 조각이 질의입니다.
3. 뒤 조각의 번호는 1입니다.


In [ ]:
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-9 · 경로 맨 뒤의 IP 꺼내기</font></h3></td></tr></table>

주소의 **경로 맨 뒤**에 있는 IP 를 꺼내 출력하시오.

- 질의를 먼저 떼어 낸 뒤 슬래시로 쪼갭니다.

| | |
|---|---|
| 주어지는 값 | `url = "https://api.example.com/v1/ip/185.220.101.34?full=true"` |
| 🎯 나와야 하는 결과 | `185.220.101.34` |

**💡 힌트**

1. 먼저 `?` 로 쪼개 앞 조각만 남깁니다.
2. 그 조각을 `/` 로 다시 쪼갭니다.
3. 쪼갠 리스트를 `parts` 라 하면 맨 뒤 번호는 `len(parts) - 1` 입니다.


In [ ]:
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-10 · 경보에 나온 IP 로 주소 만들기</font></h3></td></tr></table>

오전 룰 ②가 찾은 IP 목록을 돌면서 **조회 주소를 한 줄씩** 출력하시오.

| | |
|---|---|
| 주어지는 값 | `ips = ["185.220.101.34", "211.45.12.9"]` |
| 🎯 나와야 하는 결과 | 주소 두 줄 |

**💡 힌트**

1. 주소는 f-string 으로 만듭니다 — `f"https://api.example.com/v1/ip/{ip}?full=true"`
2. `for` 로 목록을 하나씩 돌면서 그 줄을 출력합니다.


In [ ]:
ips = ["185.220.101.34", "211.45.12.9"]


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-11 · 같은 IP 는 한 번만 조회하기</font></h3></td></tr></table>

조회할 IP 목록에 **같은 IP 가 두 번 들어 있습니다.** 중복을 없애고 주소를 만들어 `lookup_urls.txt` 로 저장하시오.
바깥 서비스는 **부른 횟수만큼 돈이 들기 때문에**, 같은 것을 두 번 묻지 않는 것이 중요합니다.

| | |
|---|---|
| 주어지는 값 | `ips = ["185.220.101.34", "211.45.12.9", "185.220.101.34"]` |
| 만들 주소 | `https://api.example.com/v1/ip/{ip}?full=true` — `{ip}` 자리에 IP 를 끼웁니다 |
| 🎯 확인 | `!cat lookup_urls.txt` 로 **두 줄만** 보인다 |

**💡 힌트**

- 이미 담은 IP 인지 먼저 확인하면 됩니다. 오전에 IP 마다 계정을 모을 때 쓴 방법과 같습니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `json.loads(글자)` | 응답 글자를 딕셔너리로 되돌린다 |
| `url.split("?")` | 주소와 질의를 가른다 |
| `url.split("/")[2]` | 호스트를 꺼낸다 |
| `GET` / `POST` | 조회한다 / 새로 만든다 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">7교시 (16:00–16:50) · 상태코드와 인증</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **상태코드** | 세 자리 숫자의 앞자리로 무엇을 알 수 있나 |
| **인증과 인가** | 둘은 어떻게 다른가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 상태코드 →
- 인증과 인가 →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6 · 답이 안 올 때를 읽는다</mark>


### 왜 필요한가

1. 요청을 보냈는데 원하는 값이 안 왔습니다. 서버가 죽은 것인지, 주소가 틀린 것인지, 권한이 없는 것인지 알아야 **다음 행동이 정해집니다.**
2. 그래서 응답에는 세 자리 숫자가 함께 옵니다. **앞자리 하나**만 봐도 어느 쪽 잘못인지 갈립니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 상태코드 | 요청이 어떻게 됐는지 알리는 세 자리 숫자 |
| 인증 | 누구인지 밝히는 일 |
| 인가 | 그 사람에게 권한이 있는지 보는 일 |
| API 키 | 발급받은 긴 글자를 요청마다 붙이는 방식 |
| 토큰 | 먼저 받아 두고 붙이는, 유효 기간이 있는 글자 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.1 상태코드 — 앞자리로 좁힌다</mark>

| 앞자리 | 뜻 | 예 |
|---|---|---|
| `2` | 성공 | `200` 처리했다 · `201` 새로 만들었다 |
| `4` | **보낸 쪽** 잘못 | `401` `403` `404` |
| `5` | **서버** 잘못 | `500` 서버가 터졌다 |

앞자리는 글자로 바꿔 첫 글자를 보면 됩니다.

```python
code = 404
print(str(code)[0])     # 4
```

`str()` 은 숫자를 글자로 바꾸는 명령입니다. **글자도 리스트처럼 번호로 꺼낼 수 있습니다** — `[0]` 은 첫 글자입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요.

```python
code = 503

print(str(code)[0])
```

막히면 바로 위 `7.1 상태코드 — 앞자리로 좁힌다` 설명을 다시 봅니다.


In [8]:
code = 503

print(str(code)[0])


5


✅ `5`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 목록을 돌면서 앞자리만 봅니다.

```python
codes = [200, 404, 500]

for code in codes:
    print(str(code)[0])
```

막히면 바로 위 `7.1 상태코드 — 앞자리로 좁힌다` 설명을 다시 봅니다.


In [9]:
codes = [200, 404, 500]

for code in codes:
    print(str(code)[0])


2
4
5


✅ `2 · 4 · 5 세 줄`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-3 · 세 갈래로 나누기</font></h3></td></tr></table>

상태코드를 받아 앞자리에 따라 셋으로 갈라 출력하시오.

| 앞자리 | 출력할 말 |
|---|---|
| `2` | `성공` |
| `4` | `내가 보낸 요청이 잘못됐다` |
| `5` | `서버 쪽 문제다` |

| | |
|---|---|
| 주어지는 값 | `code = 403` |
| 🎯 나와야 하는 결과 | `내가 보낸 요청이 잘못됐다` |

**💡 힌트**

1. 앞자리는 `str(code)[0]` 으로 꺼냅니다 — 글자 한 개가 나옵니다.
2. `if` → `elif` → `else` 로 세 갈래를 만듭니다. 견주는 값은 모두 글자입니다.
3. 처음 참이 된 갈래 하나만 실행됩니다.


In [11]:
code = 403
first = str(code)[0]

if first == "2":
    print("성공")
elif first == "4":
    print("내가 보낸 요청이 잘못됐다")
else:
    print("서버 쪽 문제다")

내가 보낸 요청이 잘못됐다


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-4 · 실패한 요청 세기</font></h3></td></tr></table>

응답 목록에서 **앞자리가 4인 것**이 몇 개인지 세어 출력하시오.

| | |
|---|---|
| 주어지는 값 | `codes = [200, 404, 500, 403, 200, 401]` |
| 🎯 나와야 하는 결과 | `보낸 쪽 잘못 3건` |

**💡 힌트**

1. 셀 숫자를 담을 이름을 반복 **전에** 만들어 0 으로 둡니다 — `fail = 0`
2. 반복 안에서 앞자리를 견줍니다.
3. 출력은 반복이 끝난 뒤 한 번만 합니다.


In [13]:
codes = [200, 404, 500, 403, 200, 401]
fail = 0

for code in codes:
    if str(code)[0] == "4":
        fail = fail + 1

print(f"보낸 쪽 잘못 {fail}건")

보낸 쪽 잘못 3건


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-5 · 200 인데 내용은 실패</font></h3></td></tr></table>

상태코드가 `200` 이어도 **내용이 실패**일 수 있습니다. **두 겹을 모두 확인**해서 출력하시오.

- 상태코드가 `200` 이고 `success` 가 참이면 → `country` 를 출력
- 둘 중 하나라도 아니면 → `조회 못 함: ` 뒤에 `message` 를 출력

| | |
|---|---|
| 주어지는 값 | `status = 200` · `text = '{"success": false, "message": "private range"}'` |
| 🎯 나와야 하는 결과 | `조회 못 함: private range` |

이 응답은 오전 로그의 내부망 IP `10.1.2.11` 을 조회한 결과입니다. `10.` 으로 시작하는 **사설 IP** 는 회사·집 안에서만 쓰는 주소라 인터넷에 소유 정보가 없습니다. 주고받기 자체는 성공해서 상태코드는 `200` 입니다.

**💡 힌트**

1. `json.loads` 로 되돌린 뒤 `data["success"]` 를 봅니다.
2. 조건 두 개를 `and` 로 묶습니다 — 룰 ③ 에서 쓴 그 방법입니다.
3. `data["success"]` 는 참·거짓이라 `== True` 를 붙이지 않아도 됩니다.


In [17]:
status =200
text = '{"success": false, "message": "private range"}'
import json

data = json.loads(text)

if status == 200 and data["success"]:
    print(data["country"])
else:
    print("조회 못 함:", data["message"])

조회 못 함: private range


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-6 · 인증 실패가 몰린 IP 찾기</font></h3></td></tr></table>

요청 기록을 돌면서 **`401` 이 3회 이상인 IP** 를 찾아 차단 검토 문구를 출력하시오.
`401` 은 「누구인지 확인이 안 됐다」는 뜻이라, 한 곳에서 몰리면 남의 계정을 찔러 보는 중일 수 있습니다.

| | |
|---|---|
| 주어지는 값 | `logs = [{"ip": "211.45.12.9", "code": 401}, {"ip": "211.45.12.9", "code": 401}, {"ip": "10.1.2.11", "code": 200}, {"ip": "211.45.12.9", "code": 401}, {"ip": "10.1.2.11", "code": 401}]` |
| 🎯 나와야 하는 결과 | `[차단 검토] 211.45.12.9 — 인증 실패 3회` |

**💡 힌트**

1. `401` 인 것만 셉니다. 세는 기준은 IP 입니다.
2. 다 센 뒤에 3 이상인 것만 출력합니다.


In [28]:
# 여기에 코드를 입력하세요counts = {}
logs = [{"ip": "211.45.12.9", "code": 401}, {"ip": "211.45.12.9", "code": 401}, {"ip": "10.1.2.11", "code": 200}, {"ip": "211.45.12.9", "code": 401}, {"ip": "10.1.2.11", "code": 401}]
counts = {}

for log in logs:
    if log["code"] == 401:
        ip = log["ip"]

        if ip not in counts:
            counts[ip] = 0

        counts[ip] = counts[ip] + 1

for ip in counts:
    if counts[ip] >= 3:
        print(f"[차단 검토] {ip} - 인증 실패 {counts[ip]}회")

[차단 검토] 211.45.12.9 - 인증 실패 3회


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 7-7 · 앞자리별로 세기</font></h3></td></tr></table>

응답 목록을 앞자리별로 세어 출력하시오.

| | |
|---|---|
| 주어지는 값 | `codes = [200, 404, 500, 403, 200, 401]` |
| 🎯 나와야 하는 결과 | `2 — 2건` · `4 — 3건` · `5 — 1건` |

**💡 힌트**

- 앞자리만 남기면 같은 값끼리 묶입니다. 세는 방법은 오전에 계정을 셀 때와 같습니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.2 401 ↔ 403 · 인증 두 방식</mark>

둘 다 막히는 상황이지만 **원인이 다릅니다.**

| 견주는 점 | `401` | `403` |
|---|---|---|
| 서버가 하는 말 | 당신이 누구인지 **모르겠다** | 누구인지는 알겠는데 **안 된다** |
| 어디서 막혔나 | 인증 | 인가 |
| 할 일 | 토큰을 다시 받아 붙인다 | 담당자에게 권한을 요청한다 |

누구인지 밝히는 방식은 크게 둘입니다.

| 방식 | 어떻게 | 약점 |
|---|---|---|
| API 키 | 발급받은 긴 글자를 요청마다 붙인다 | 한 번 새면 바꾸기 전까지 계속 쓰인다 |
| 토큰 | 먼저 받아 두고 붙인다 | 유효 기간이 있어 피해가 시간에 갇힌다 |

붙이는 자리는 **헤더**입니다. `Authorization: Bearer 토큰` 꼴로 적습니다.

> **키를 코드 밖에 두는 방법(환경변수)은 내일 배웁니다.** 실제로 키를 쓰는 것이 내일이기 때문입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-8 · 목록에 할 일 붙이기</font></h3></td></tr></table>

응답 목록을 돌면서 코드마다 **할 일**을 한 줄씩 출력하시오.

| | |
|---|---|
| 주어지는 값 | `codes = [200, 401, 403, 500]` |
| 🎯 나와야 하는 결과 | `200 그대로 쓴다` · `401 토큰을 다시 받는다` · `403 권한을 요청한다` · `500 잠시 뒤 다시 보낸다` |

**💡 힌트**

1. 코드를 키, 할 일을 값으로 하는 딕셔너리를 먼저 만듭니다 — `401` 은 토큰을 다시 받고 `403` 은 권한을 요청합니다.
2. `for` 로 목록을 돌면서 있는 코드인지 `if code in todo:` 로 확인합니다.
3. 없는 코드는 그대로 쓴다고 적습니다.


In [29]:
codes = [200, 401, 403, 500]



<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-9 · 요청에 붙일 헤더 만들기</font></h3></td></tr></table>

토큰을 받아 요청에 붙일 **헤더 딕셔너리**를 만들어 출력하시오.

| | |
|---|---|
| 주어지는 값 | `token = "sk-0929-demo"` |
| 🎯 나와야 하는 결과 | `{'Authorization': 'Bearer sk-0929-demo'}` |

**💡 힌트**

1. 칸 이름은 `Authorization` 입니다.
2. 값은 `Bearer ` 뒤에 토큰을 이어 붙인 글자입니다.
3. 글자끼리는 `+` 로 이어 붙입니다. f-string 으로 해도 됩니다.


In [ ]:
token = "sk-0929-demo"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-10 · 401 이면 한 번만 다시 보내기</font></h3></td></tr></table>

응답이 `401` 이면 **토큰을 새로 받아 한 번만** 다시 보내고, 그 결과를 출력하시오.
토큰은 시간이 지나면 만료됩니다. 실무에서는 이렇게 **한 번만** 다시 보냅니다 — 계속 다시 보내면 상대 서버를 두드리는 꼴이 됩니다.

| | |
|---|---|
| 주어지는 값 | `answer = {"code": 401, "body": "token expired"}` · 다시 보낸 뒤의 응답은 `{"code": 200, "body": "ok"}` 라고 칩니다 |
| 🎯 나와야 하는 결과 | 다시 보냈다는 줄과 `요청 성공` |

**💡 힌트**

1. `401` 인지 먼저 확인하고, 그 안에서 응답을 새 값으로 바꿉니다.
2. 바꾼 뒤에 다시 `200` 인지 확인해 결과를 출력합니다.
3. `for` 는 필요 없습니다. **한 번만** 다시 보내는 것이 핵심입니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 7-11 · 조회 결과를 경보에 붙이기</font></h3></td></tr></table>

상태코드와 응답을 함께 받아, **성공일 때만** 위험 등급을 경보에 붙여 출력하시오.

| | |
|---|---|
| 주어지는 값 | `results = [{"ip": "185.220.101.34", "code": 200, "body": '{"country": "Germany"}'}, {"ip": "211.45.12.9", "code": 403, "body": ""}]` |
| 🎯 나와야 하는 결과 | `[조회] 185.220.101.34 — Germany` 와 `[조회 실패] 211.45.12.9 — 코드 403` |

**💡 힌트**

- 성공인지 먼저 보고, 성공일 때만 내용을 읽으면 됩니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `str(code)[0]` | 상태코드 앞자리 |
| `2` / `4` / `5` | 성공 / 보낸 쪽 잘못 / 서버 잘못 |
| `401` | 누구인지 모르겠다 → 토큰을 다시 |
| `403` | 누구인지는 알겠는데 안 된다 → 권한 요청 |
| `{"Authorization": "Bearer " + token}` | 요청에 붙이는 헤더 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">8교시 (17:00–17:30) · 마무리</mark>


새로 배우는 것이 없습니다. 오늘 만든 것을 확인하고 마무리합니다.

### 8.1 오늘 만든 것

| 파일 | 무엇 | 언제 |
|---|---|---|
| `raw_logs.txt` | 쉼표가 없는 서버 로그 원본 | 오전 2교시 |
| `normalize_logs.py` | 정규식으로 한 모양으로 맞추는 프로그램 | 오전 3교시 |
| `normalized_logs.json` | 정규화한 기록 16건 | 오전 3교시 |
| `unmatched_logs.txt` | 규칙에 안 맞은 줄 4건 | 오전 3교시 |

드라이브 `agent_core` 폴더를 열어 네 개가 다 있는지 봅니다. 없으면 오전 문제 4-10 으로 돌아가 다시 합니다.


### 8.2 경보는 왜 파일로 안 남겼나

오늘 만든 탐지 룰 세 개는 화면에만 찍었습니다. 학원 교안이 정한 오늘 산출물은 위의 네 개뿐이고, **교안에 없는 이름을 우리가 새로 짓지 않기** 때문입니다.

내일 조회할 IP 가 필요해지면 **`normalized_logs.json` 을 다시 읽어 룰 ②를 한 번 더 돌리면 됩니다.** 오늘 문제 5-10 가 그 코드입니다.


### 8.3 남은 ⭐도전 문제

넘어간 도전 문제를 여기서 풉니다. **다 풀지 않아도 됩니다.** 기본 문제를 못 끝낸 분은 도전이 아니라 기본부터 합니다.

| 어디 | 도전 문제 |
|---|---|
| 오전 2교시 | ⭐1-1 계정이 아닌 줄 세기 · ⭐1-2 WARN 인 줄 세기 |
| 오전 3교시 | ⭐2-1 번호 바꿔 보기 · ⭐2-2 계정만 모아 보기 · ⭐2-3 안 맞는 줄 읽기 |
| 오전 4교시 | ⭐3-1 임계값 바꿔 보기 · ⭐3-2 두 룰을 한 번에 |
| 오후 5교시 | ⭐4-1 기준을 바꿔 보기 · ⭐4-2 어느 룰이 몇 건인가 |
| 오후 6교시 | ⭐5-1 응답 세 개를 한 번에 · ⭐5-2 조회 목록을 파일로 |
| 오후 7교시 | ⭐6-1 앞자리별로 세기 · ⭐6-2 조회 결과를 경보에 붙이기 |


### 8.4 오늘 배운 것 한눈에

| 시간 | 배운 것 |
|---|---|
| 오전 | `split` 이 조용히 틀린다 · `re.search` 와 기호 · 그룹과 `.groupdict()` · 안 맞는 줄 남기기 · 탐지 룰 ①② |
| 오후 | 탐지 룰 ③ 심야 성공 · 요청과 응답 · 주소 쪼개 읽기 · 상태코드 · `401` ↔ `403` |

**내일(9/30) 오전**에는 오늘 흉내만 낸 요청을 `requests` 로 실제로 보냅니다. API 키를 코드 밖에 두는 법도 그때 배웁니다.


---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 엽니다</mark>

각 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다.


In [ ]:
#@title 정답 5-3 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

for row in rows:
    hour = int(row["time"].split(":")[0])
    if hour < 6 and row["level"] == "INFO":
        print(f"[룰 3] 심야 접속: {row['time']} {row['user']} ({row['ip']})")


In [ ]:
#@title 정답 5-4 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

NIGHT_END = 6

for row in rows:
    hour = int(row["time"].split(":")[0])
    if hour < NIGHT_END and row["level"] == "INFO":
        print(f"[룰 3] 심야 접속: {row['time']} {row['user']} ({row['ip']})")


In [ ]:
#@title 정답 5-5 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

count = {}
for row in rows:
    if row["level"] == "WARN":
        hour = row["time"].split(":")[0]
        key = row["user"] + " " + hour + "시"
        if key in count:
            count[key] = count[key] + 1
        else:
            count[key] = 1

for key in count:
    if count[key] >= 3:
        print(f"[집중 실패] {key} — {count[key]}회")


In [ ]:
#@title 정답 ⭐5-6 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

for night_end in [6, 9, 12]:
    hit = 0
    for row in rows:
        hour = int(row["time"].split(":")[0])
        if hour < night_end and row["level"] == "INFO":
            hit = hit + 1
    print(f"기준 {night_end}시 — 경보 {hit}건")

# 근거 예시: 9시나 12시로 넓히면 출근 시간 정상 접속까지 잡혀 헛경보가 는다.
# 새벽만 보려는 것이 목적이니 6시로 둔다.


In [ ]:
#@title 정답 5-7 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

THRESHOLD = 3
NIGHT_END = 6

count = {}
table = {}

for row in rows:
    user = row["user"]
    ip = row["ip"]
    if row["level"] == "WARN":
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1
    if ip in table:
        if user not in table[ip]:
            table[ip].append(user)
    else:
        table[ip] = [user]

for user in count:
    if count[user] >= THRESHOLD:
        print(f"[룰 1] 확인 필요: {user} — 실패 {count[user]}회")

for ip in table:
    if len(table[ip]) >= 2:
        print(f"[룰 2] 의심 IP: {ip} — 계정 {len(table[ip])}개 시도")

for row in rows:
    if int(row["time"].split(":")[0]) < NIGHT_END and row["level"] == "INFO":
        print(f"[룰 3] 심야 접속: {row['time']} {row['user']} ({row['ip']})")


In [ ]:
#@title 정답 5-8 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

THRESHOLD = 3
NIGHT_END = 6

count = {}
table = {}

for row in rows:
    user = row["user"]
    ip = row["ip"]
    if row["level"] == "WARN":
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1
    if ip in table:
        if user not in table[ip]:
            table[ip].append(user)
    else:
        table[ip] = [user]

alerts = []

for user in count:
    if count[user] >= THRESHOLD:
        alerts.append(f"[룰 1] 확인 필요: {user} — 실패 {count[user]}회")

for ip in table:
    if len(table[ip]) >= 2:
        alerts.append(f"[룰 2] 의심 IP: {ip} — 계정 {len(table[ip])}개 시도")

for row in rows:
    if int(row["time"].split(":")[0]) < NIGHT_END and row["level"] == "INFO":
        alerts.append(f"[룰 3] 심야 접속: {row['time']} {row['user']} ({row['ip']})")

print(f"[요약] 정규화 {len(rows)}건 / 경보 {len(alerts)}건")
for line in alerts:
    print(line)


In [ ]:
#@title 정답 5-9 { display-mode: "form" }
targets = ["admin", "185.220.101.34", "admin"]

count = {}
for name in targets:
    if name in count:
        count[name] = count[name] + 1
    else:
        count[name] = 1

for name in count:
    if count[name] >= 2:
        print(f"[HIGH] {name} — {count[name]}개 룰")
    else:
        print(f"[MEDIUM] {name} — {count[name]}개 룰")


In [ ]:
#@title 정답 ⭐5-10 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

THRESHOLD = 3
NIGHT_END = 6

# 룰 ① — 계정별 실패를 세어 기준을 넘은 계정을 모은다
fail_count = {}
for row in rows:
    if row["level"] == "WARN":
        user = row["user"]
        if user in fail_count:
            fail_count[user] = fail_count[user] + 1
        else:
            fail_count[user] = 1

rule1_users = []
for user in fail_count:
    if fail_count[user] >= THRESHOLD:
        rule1_users.append(user)

# 룰 ③ — 새벽에 성공한 계정을 모은다
rule3_users = []
for row in rows:
    hour = int(row["time"].split(":")[0])
    if hour < NIGHT_END and row["level"] == "INFO":
        if row["user"] not in rule3_users:
            rule3_users.append(row["user"])

print("룰 1 이 내놓은 계정:", rule1_users)
print("룰 3 이 내놓은 계정:", rule3_users)

for user in rule1_users:
    if user in rule3_users:
        print(f"두 룰에 함께 걸림: {user}")


In [ ]:
#@title 정답 6-3 { display-mode: "form" }
import json

response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'

data = json.loads(response_text)

print(data["country"])


In [ ]:
#@title 정답 6-4 { display-mode: "form" }
import json

response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'

data = json.loads(response_text)

if data["country_code"] != "KR":
    print(f"[조회] {data['ip']} — 해외 접속 ({data['country']})")


In [ ]:
#@title 정답 6-5 { display-mode: "form" }
import json

response_text = '{"ip": "185.220.101.34", "country": "Germany", "country_code": "DE", "city": "Berlin", "asn": 60729}'

HOME = "KR"
data = json.loads(response_text)

if data["country_code"] == HOME:
    print("국내 접속")
else:
    print("해외 접속 — 확인 필요")


In [ ]:
#@title 정답 6-6 { display-mode: "form" }
import json

answers = ['{"ip": "185.220.101.34", "country": "Germany", "city": "Berlin"}',
           '{"ip": "8.8.8.8", "country": "United States"}']

for text in answers:
    data = json.loads(text)
    if "city" in data:
        print(f"{data['ip']} — {data['city']}")
    else:
        print(f"{data['ip']} — 도시 정보 없음")


In [ ]:
#@title 정답 ⭐6-7 { display-mode: "form" }
import json

responses = [
    '{"ip": "185.220.101.34", "country_code": "DE"}',
    '{"ip": "10.1.2.11", "country_code": "KR"}',
    '{"ip": "203.0.113.5", "country_code": "US"}',
]

for text in responses:
    data = json.loads(text)
    if data["country_code"] != "KR":
        print(data["ip"])


In [ ]:
#@title 정답 6-8 { display-mode: "form" }
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"

print(url.split("?")[1])


In [ ]:
#@title 정답 6-9 { display-mode: "form" }
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"

address = url.split("?")[0]
parts = address.split("/")

print(parts[len(parts) - 1])


In [ ]:
#@title 정답 6-10 { display-mode: "form" }
ips = ["185.220.101.34", "211.45.12.9"]

for ip in ips:
    url = f"https://api.example.com/v1/ip/{ip}?full=true"
    print(url)


In [ ]:
#@title 정답 ⭐6-11 { display-mode: "form" }
ips = ["185.220.101.34", "211.45.12.9", "185.220.101.34"]

done = []
for ip in ips:
    if ip not in done:
        done.append(ip)

with open("lookup_urls.txt", "w", encoding="utf-8") as f:
    for ip in done:
        f.write(f"https://api.example.com/v1/ip/{ip}?full=true" + chr(10))

print(f"{len(done)}줄을 저장했습니다")


In [ ]:
#@title 정답 7-3 { display-mode: "form" }
code = 403

first = str(code)[0]

if first == "2":
    print("성공")
elif first == "4":
    print("내가 보낸 요청이 잘못됐다")
else:
    print("서버 쪽 문제다")


In [ ]:
#@title 정답 7-4 { display-mode: "form" }
codes = [200, 404, 500, 403, 200, 401]

bad = 0

for code in codes:
    if str(code)[0] == "4":
        bad = bad + 1

print(f"보낸 쪽 잘못 {bad}건")


In [ ]:
#@title 정답 7-5 { display-mode: "form" }
import json

status = 200
text = '{"success": false, "message": "private range"}'

data = json.loads(text)

if status == 200 and data["success"]:
    print(data["country"])
else:
    print("조회 못 함:", data["message"])


In [ ]:
#@title 정답 7-6 { display-mode: "form" }
logs = [{"ip": "211.45.12.9", "code": 401},
        {"ip": "211.45.12.9", "code": 401},
        {"ip": "10.1.2.11", "code": 200},
        {"ip": "211.45.12.9", "code": 401},
        {"ip": "10.1.2.11", "code": 401}]

count = {}
for row in logs:
    if row["code"] == 401:
        ip = row["ip"]
        if ip in count:
            count[ip] = count[ip] + 1
        else:
            count[ip] = 1

for ip in count:
    if count[ip] >= 3:
        print(f"[차단 검토] {ip} — 인증 실패 {count[ip]}회")


In [ ]:
#@title 정답 ⭐7-7 { display-mode: "form" }
codes = [200, 404, 500, 403, 200, 401]

count = {}

for code in codes:
    first = str(code)[0]
    if first in count:
        count[first] = count[first] + 1
    else:
        count[first] = 1

for first in count:
    print(f"{first} — {count[first]}건")


In [ ]:
#@title 정답 7-8 { display-mode: "form" }
codes = [200, 401, 403, 500]

todo = {
    200: "그대로 쓴다",
    401: "토큰을 다시 받는다",
    403: "권한을 요청한다",
    500: "잠시 뒤 다시 보낸다",
}

for code in codes:
    if code in todo:
        print(code, todo[code])
    else:
        print(code, "알 수 없음")


In [ ]:
#@title 정답 7-9 { display-mode: "form" }
token = "sk-0929-demo"

headers = {"Authorization": "Bearer " + token}

print(headers)


In [ ]:
#@title 정답 7-10 { display-mode: "form" }
answer = {"code": 401, "body": "token expired"}
token = "sk-0929-demo"

if answer["code"] == 401:
    token = "sk-0929-new"
    answer = {"code": 200, "body": "ok"}
    print(f"401 이라 토큰을 새로 받아 한 번 더 보냈습니다 — 새 코드 {answer['code']}")

if answer["code"] == 200:
    print("요청 성공")
else:
    print("다시 보내도 실패했습니다")


In [ ]:
#@title 정답 ⭐7-11 { display-mode: "form" }
import json

results = [
    {"ip": "185.220.101.34", "code": 200, "body": '{"country": "Germany"}'},
    {"ip": "211.45.12.9", "code": 403, "body": ""},
]

for result in results:
    if str(result["code"])[0] == "2":
        data = json.loads(result["body"])
        print(f"[조회] {result['ip']} — {data['country']}")
    else:
        print(f"[조회 실패] {result['ip']} — 코드 {result['code']}")
